# Linux for AI Engineer

NLP, ML, AI နဲ့ ပတ်သက်တဲ့ အလုပ်တွေအားလုံးလိုလိုကို Linux OS ပေါ်မှာပဲ လုပ်ကြတယ်။ အကြောင်းအရင်းကတော့ Open source ဖြစ်တာကော၊ security ပိုင်းအရကော၊ Linux command တွေက powerful ဖြစ်တာကြောင့်ကော စသည်ဖြင့် အမျိုးမျိုးပါပဲ။ အဲဒါကြောင့် ဒီနေ့မှာတော့ မသိမဖြစ်တဲ့ Linux command တချို့ကို လက်တွေ့လုပ်ပြရင်း မိတ်ဆက်ပေးပါမယ်။   

ရဲကျော်သူ  
Language Understanding Lab., Myanmar  
15 Mar 2026  

## Preparation

To give you a demo, let's build a mini-environment. You can run these one by one in your terminal.
First, let's create our "lab data." Copy and paste this entire block into your terminal to generate the files we need:  

ကော်ပီကူးတဲ့အခါမှာ markdown သင်္ကေတ ` ``` ` တွေတော့ ဖြုတ်ကူးပါ။  

# Create a dummy dataset

```bash
cat <<EOF > dataset.csv
id,text,label,confidence
1,"I love this transformer model",pos,0.98
2,"The training is extremely slow",neg,0.85
3,"Great accuracy on the test set",pos,0.92
4,"The model is hallucinating",neg,0.76
5,"This is an okay result",neu,0.50
6,"Fantastic performance",pos,0.99
EOF
``` 

# Create a fake training log

```bash
cat <<EOF > train.log
epoch: 1 | loss: 0.85 | accuracy: 0.55
epoch: 2 | loss: 0.65 | accuracy: 0.70
epoch: 3 | loss: 0.45 | accuracy: 0.85
[ERROR] GPU Out of Memory at epoch 4
EOF
```

# Create a dummy Python script

```bash
cat <<EOF > train.py
import time
print("Training started...")
time.sleep(100) # Simulating a long process
EOF
```

## 1. wc (Word Count)

- The Task: "How many data samples do I have?"

```bash
wc -l dataset.csv
```

- Why it matters: Quickest way to verify if your dataset downloaded correctly or if you lost rows during processing.

In [4]:
!wc -l dataset.csv

7 dataset.csv


In [5]:
!wc dataset.csv

  7  28 267 dataset.csv


## 2. head and tail

- The Task: "Show me the top 3 rows and the last 2 lines of the log."

```bash
head -n 3 dataset.csv
tail -n 2 train.log
```

- Why it matters: Use head to check headers and tail to see the most recent training errors or loss values.

In [7]:
!head -n 3 dataset.csv

id,text,label,confidence
1,"I love this transformer model",pos,0.98
2,"The training is extremely slow",neg,0.85


In [8]:
!tail -n 2 train.log

epoch: 3 | loss: 0.45 | accuracy: 0.85
[ERROR] GPU Out of Memory at epoch 4


## 3. grep (The Search Engine)

- The Task: "Find all 'negative' samples in my data."

```bash
grep "neg" dataset.csv
grep -n "neg" dataset.csv
```

- Tip: Use ```grep -c "pos" dataset.csv``` to count how many positive samples exist without listing them.

In [9]:
!grep "neg" dataset.csv

2,"The training is extremely slow",neg,0.85
4,"The model is hallucinating",neg,0.76


In [10]:
!grep -n "neg" dataset.csv

3:2,"The training is extremely slow",neg,0.85
5:4,"The model is hallucinating",neg,0.76


In [13]:
!grep -c "pos" dataset.csv

3


## 4. cut (Column Extraction)

- The Task: "I only want to see the text column (column 2)."

```bash
cut -d',' -f2 dataset.csv
```

- Why it matters: Often we don't need IDs or timestamps; cut lets you isolate features or labels instantly.

In [14]:
!cut -d',' -f2 dataset.csv

text
"I love this transformer model"
"The training is extremely slow"
"Great accuracy on the test set"
"The model is hallucinating"
"This is an okay result"
"Fantastic performance"


## 5. sort and uniq

- The Task: "Show me the distribution of my labels."

``` bash
cut -d',' -f3 dataset.csv | tail -n +2 | sort | uniq -c
``` 

- Breakdown: ```tail -n +2``` skips the header line.  
-- sort groups identical labels together.  
-- uniq -c counts the occurrences of each unique label.  

In [15]:
!cut -d',' -f3 dataset.csv | tail -n +2 | sort | uniq -c

      2 neg
      1 neu
      3 pos


Linux မှာက command တွေက လေ့လာလို့ မကုန်နိုင်ပါဘူး။ built-in command မဟုတ်တာတွေကိုတော့ sudo right ရှိရင် Ubuntu မှာ ဆိုရင် apt-get ဆိုတဲ့ command မျိုးနဲ့ install လုပ်ယူနိုင်ပါတယ်။ ဥပမာ multitail ဆိုတဲ့ command ကို install လုပ်ပြီး သုံးမယ် ဆိုရင် အောက်ပါအတိုင်း။  

```bash
ye@lst-hpc3090:~/aif/linux$ sudo apt-get install multitail
[sudo] password for ye: 
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  multitail
0 upgraded, 1 newly installed, 0 to remove and 30 not upgraded.
Need to get 130 kB of archives.
After this operation, 344 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 multitail amd64 6.5.0-6 [130 kB]
Fetched 130 kB in 1s (110 kB/s)     
Selecting previously unselected package multitail.
(Reading database ... 397608 files and directories currently installed.)
Preparing to unpack .../multitail_6.5.0-6_amd64.deb ...
Unpacking multitail (6.5.0-6) ...
Setting up multitail (6.5.0-6) ...
Processing triggers for man-db (2.12.0-4build2) ...
ye@lst-hpc3090:~/aif/linux$ ls
dataset.csv  Linux-intro.ipynb  train.log  train.py
ye@lst-hpc3090:~/aif/linux$ multitail -n 2 train.log train.py
```

## 6. sed (Search and Replace)

- The Task: "Change 'pos' to 'Positive' throughout the file."

```bash
sed 's/pos/Positive/g' dataset.csv
``` 

- Why it matters: Perfect for fixing typos in labels or cleaning up HTML tags in a dataset without opening a heavy editor.

In [16]:
!sed 's/pos/Positive/g' dataset.csv

id,text,label,confidence
1,"I love this transformer model",Positive,0.98
2,"The training is extremely slow",neg,0.85
3,"Great accuracy on the test set",Positive,0.92
4,"The model is hallucinating",neg,0.76
5,"This is an okay result",neu,0.50
6,"Fantastic performance",Positive,0.99


In [17]:
!sed 's/pos/Positive/g;s/neg/Negative/g' dataset.csv

id,text,label,confidence
1,"I love this transformer model",Positive,0.98
2,"The training is extremely slow",Negative,0.85
3,"Great accuracy on the test set",Positive,0.92
4,"The model is hallucinating",Negative,0.76
5,"This is an okay result",neu,0.50
6,"Fantastic performance",Positive,0.99


အထက်မှာ ပြထားတဲ့ command တွေကို run လိုက်ရင် အစားထိုးထားတဲ့ output ကို screen ပေါ်မှာပဲ ပြပေးတာ။  
ကိုယ်က ဖိုင်အသစ် တစ်ဖိုင်အနေနဲ့ သိမ်းစေချင်တယ် ဆိုရင် redirect သင်္ကေတ ဖြစ်တဲ့ ">" နဲ့ stdout ကို ကိုယ် အသစ်ရေးပေးစေချင်တဲ့ ဖိုင်နာမည်ပေးပြီး run ပါ။  

**သတိထားရမှာက ရှိပြီးသား ဖိုင်နာမည်ကို ပေးလိုက်ရင် အော်ရဂျင်နယ်ဖိုင်ပေါ်ကို ထပ်ရေးလိုက်ပါလိမ့်မယ်**

In [18]:
!sed 's/pos/Positive/g;s/neg/Negative/g' dataset.csv > dataset-long.csv

In [19]:
!cat dataset-long.csv

id,text,label,confidence
1,"I love this transformer model",Positive,0.98
2,"The training is extremely slow",Negative,0.85
3,"Great accuracy on the test set",Positive,0.92
4,"The model is hallucinating",Negative,0.76
5,"This is an okay result",neu,0.50
6,"Fantastic performance",Positive,0.99


တကယ်လို့ ရှိပြီးသားဖိုင်ကိုမှ ဝင်ပြင်ချင်တယ် ဆိုရင် sed commnad ရဲ့ command line option ဖြစ်တဲ့ "-i" ကို သုံးပါ။  
**သတိတော့ထားပါ**

In [20]:
!sed -i 's/Positive/pos/g;s/Negative/neg/g' dataset-long.csv

In [21]:
!cat dataset-long.csv

id,text,label,confidence
1,"I love this transformer model",pos,0.98
2,"The training is extremely slow",neg,0.85
3,"Great accuracy on the test set",pos,0.92
4,"The model is hallucinating",neg,0.76
5,"This is an okay result",neu,0.50
6,"Fantastic performance",pos,0.99


ရှာတွေ့ပြီး ပြင်သွားတဲ့ လိုင်းတွေကိုပဲ print ရိုက်ထုတ်ပေးစေချင်ရင် "-n" option နဲ့ တွဲပြီး "p" flag ကို သုံးပါ။ ဥပမာ အောက်ပါအတိုင်း  

In [25]:
!sed -n 's/pos/Positive/p;s/neg/Negative/p' dataset.csv

1,"I love this transformer model",Positive,0.98
2,"The training is extremely slow",Negative,0.85
3,"Great accuracy on the test set",Positive,0.92
4,"The model is hallucinating",Negative,0.76
6,"Fantastic performance",Positive,0.99


အသုံးပြုတဲ့ pattern အနေနဲ့က အောက်ပါအတိုင်း ပါ။  

```
sed -n 's/pattern/replacement/p' file
``` 


## 7. awk (The Mini-Programmer)

- The Task: "Find rows where confidence score (column 4) is greater than 0.90."

```bash
awk -F',' '$4 > 0.90 {print $0}' dataset.csv
```

- Why it matters: awk is like a lightweight Python. It understands columns and numbers, making it the best tool for filtering datasets based on thresholds.

In [26]:
!awk -F',' '$4 > 0.90 {print $0}' dataset.csv

id,text,label,confidence
1,"I love this transformer model",pos,0.98
3,"Great accuracy on the test set",pos,0.92
6,"Fantastic performance",pos,0.99


## 8. nvidia-smi (NVIDIA System Management Interface)

Deep-learning modeling, Neural Network based modeling တွေအတွက် မသိမဖြစ်တဲ့ command တစ်ခုပါပဲ။  
မော်ဒယ် တစ်ခု မဆောက်ခင်မှာ အမြဲ run ဖြစ်တဲ့ command ပါ။  

- The Task: "Is my GPU being used?"

```
nvidia-smi
```

- Note: This only works if you have an NVIDIA GPU and drivers installed. It shows memory usage, temperature, and power consumption.

In [27]:
!nvidia-smi

Sun Mar 15 06:44:57 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090 Ti     Off | 00000000:01:00.0  On |                  Off |
|  0%   54C    P8              31W / 480W |    415MiB / 24564MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

အထက် က ဇယားမှာ အပေါ်ဘက်က GPU နဲ့ ပတ်သက်တဲ့ summary ဖြစ်ပြီးတော့၊ အောက်ဘက်က တခြမ်းကတော့ လက်ရှိ စက်မှာ run နေတဲ့ active process တွေကို ပြသပေးတာပါ။  

GPU နဲ့ပတ်သက်ပြီး အဓိကထား ဖတ်ရမှာက အောက်ပါအတိုင်းပါ။  


- Temperature (Temp): Current GPU core temperature in Celsius.
- Memory Usage (Memory-Usage): Shows Used Memory / Total Memory.
- GPU Utilization (GPU-Util): Shows the percentage of time the GPU was busy processing tasks over the last sample period (0–100%).
- Power Usage (Pwr: Usage/Cap): Current power consumption compared to maximum capacity (Watts).


## 9. ps and kill

- The Task: "My Python script is stuck. I need to stop it."

```
    Command: 1.  ps aux | grep train.py (Find the Process ID / PID)
    Command: 2.  kill -9 [PID] (Force stop it)
```

- Why it matters: When Jupyter notebooks or training scripts hang and hog all your VRAM, this is the "Emergency Exit."

အရင်ဆုံး simulation လုပ်ကြည့်ရအောင်။  

```bash
ye@lst-hpc3090:~/aif/linux$ python ./train.py
Training started...
```

နောက်ထပ် terminal တစ်ခုကို ဖွင့်ပြီး train.py ရဲ့ Process ID (PID) ကို ရှာကြည့်ပါ။  

```bash
ye@lst-hpc3090:~/aif/linux$ ps aux | grep train.py
ye        218051  0.0  0.0  20704 10504 pts/2    S+   06:59   0:00 python ./train.py
ye        218073  0.0  0.0   9280  2320 pts/3    S+   07:00   0:00 grep --color=auto train.py
ye@lst-hpc3090:~/aif/linux$
```

train.py ပရိုဂရမ်က training လုပ်နေတာ ပြီးသင့်တဲ့အချိန်မှာ မပြီးပဲ အရမ်းကြာနေလို့ debugging လုပ်ဖို့အတွက် ရပ်လိုက်ချင်လို့ အထက်က ရှာထားတဲ့ PID နဲ့ kill လုပ်လိုက်မယ်။  

```bash
ye@lst-hpc3090:~/aif/linux$ kill -9 218051
```

## --help

ကိုယ် သုံးမယ့် Linux command ရဲ့ option တွေကို သိချင်ရင် "--help" နဲ့ ခေါ်ကြည့်ပါ။  

In [28]:
!ps --help


Usage:
 ps [options]

 Try 'ps --help <simple|list|output|threads|misc|all>'
  or 'ps --help <s|l|o|t|m|a>'
 for additional help text.

For more details see ps(1).


In [29]:
!kill --help

kill: kill [-s sigspec | -n signum | -sigspec] pid | jobspec ... or kill -l [sigspec]
    Send a signal to a job.
    
    Send the processes identified by PID or JOBSPEC the signal named by
    SIGSPEC or SIGNUM.  If neither SIGSPEC nor SIGNUM is present, then
    SIGTERM is assumed.
    
    Options:
      -s sig	SIG is a signal name
      -n sig	SIG is a signal number
      -l	list the signal names; if arguments follow `-l' they are
    		assumed to be signal numbers for which names should be listed
      -L	synonym for -l
    
    Kill is a shell builtin for two reasons: it allows job IDs to be used
    instead of process IDs, and allows processes to be killed if the limit
    on processes that you can create is reached.
    
    Exit Status:
    Returns success unless an invalid option is given or an error occurs.


## 10. nohup (No Hang Up)

- The Task: "I need to start training and then go home/close my laptop."

```
nohup python3 train.py &
```
    
- Why it matters: Usually, if you close your terminal, the code stops. nohup keeps it running in the background, and & gives you your terminal back so you can keep working.

## 11. find

- The Task: "Where did I save that .pkl model file?"

```bash
find . -name "*.pth"
```

- Why it matters: In large projects with many subfolders, find saves you from clicking through directories manually.
- Note: `.pth` and `.pkl` are file extensions commonly used in Python, particularly within the machine learning and data science ecosystem. They both serve as containers for serialized data, meaning they convert complex Python objects into a byte stream that can be saved to disk, moved, and later reconstructed (deserialized) back into a live object.

In [33]:
!pwd

/home/ye/aif/linux


In [34]:
!cd ..

In [40]:
!pwd

/home/ye/aif/linux


In [35]:
!ls

dataset.csv  Linux-intro.ipynb	train.log  train.py


In [36]:
!cd ..

In [37]:
!ls

dataset.csv  Linux-intro.ipynb	train.log  train.py


အထက်မှာ မြင်ရတဲ့အတိုင်းပဲ လက်ရှိ Jupyter notebook ရဲ့ root directory က `/home/ye/aif/linux` ဖြစ်နေလို့ cd `..` command က အလုပ် မလုပ်တော့ဘူး။ အဲဒါကြောင့် full path ပေးပြီး `%` ရှေ့မှာခံပြီးမှ folder change ရလိမ့်မယ်။ ဥပမာ။။  

In [44]:
%cd /home/ye/aif/eliza

/home/ye/aif/eliza


ခုဆိုရင် လက်ရှိ path က မနေ့က demo run ပြခဲ့တဲ့ hybrid-ELIZA ပရိုဂရမ်တွေရှိတဲ့ နေရာကို ရောက်သွားပြီ။  

In [45]:
!pwd

/home/ye/aif/eliza


In [46]:
!ls

archi  hybrid-eliza


hybrid-elizer.py ပရိုဂရမ်နဲ့ training လုပ်ပြီး ထွက်လာတဲ့ မော်ဒယ်ဖိုင်ကို ရှာချင်ရင် အောက်ပါအတိုင်း ရှာလို့ရတယ်။ ဘယ်ဖိုလ်ဒါအောက်မှာ ရှိတယ် ဆိုတာကို ပြပေးပါလိမ့်မယ်။ 

In [48]:
!find . -name "*.pth"

./hybrid-eliza/eliza_eq.pth


In [50]:
!ls -hl ./hybrid-eliza/eliza_eq.pth

-rw-rw-r-- 1 ye ye 3.0M Mar 14 13:06 ./hybrid-eliza/eliza_eq.pth


## Summary

- NLP/ML/AI-Engineering အလုပ် ၉၀% လောက်က Linux OS ပေါ်မှာ လုပ်ကြတာမို့ Linux command တွေကို ကျွမ်းကျင်ကြရလိမ့်မယ်
- မနေ့က ပေးထားတဲ့ Assignment လည်း ရှိတာမို့ ပထမဆုံး စပြီး လုပ်ကြည့်ကြမယ့် ကျောင်းသား/သူ တွေအတွက် အဆင်ပြေအောင်လို့ အရေးကြီးတဲ့ command တချို့ကို သင်ကြားပေးခဲ့တာ

## References

ဆရာ မြန်မာပြည်က နည်းပညာ တက္ကသိုလ်တွေ၊ ကွန်ပျူတာ တက္ကသိုလ်တွေမှာ စာသင်တုန်းက ပြင်ဆင်ပေးခဲ့တဲ့ Linux command Jupyter notebook ကိုလည်း လေ့လာနိုင်ရင် လေ့လာကြည့်ကြပါ။  

- [https://github.com/ye-kyaw-thu/Tutorials/blob/master/Linux-Commands/linux-commands.ipynb](https://github.com/ye-kyaw-thu/Tutorials/blob/master/Linux-Commands/linux-commands.ipynb)
- [https://github.com/ye-kyaw-thu/Tutorials/blob/master/Linux-Commands/linux-commands.pdf](https://github.com/ye-kyaw-thu/Tutorials/blob/master/Linux-Commands/linux-commands.pdf)  